<a href="https://colab.research.google.com/github/mg-de-j/MT-2025-2026/blob/main/MT_LabW3_%5BPART_2%5D_Transformer_Train_FineTune_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1> <font color="blue"> <b> Machine Translation [LIX032B05]: Lab week 3 [PART 2]</b></></h1>

Dr. Arianna Bisazza, Dr. Huiyuan Lai, and Jelmer Top

BSc Information Science, University of Groningen, February 2025

<hr width=170% /><br />

This notebook consists of <b>one part</b>:
<ul>
<li>Fine-tuning a translation model</li>
</ul>

PART 1 of the notebook gave you an insight into the steps that correspond to training a translation system using Fairseq on Tatoeba data that we also used in week 2. Then, we demonstrated how to load the pre-trained model. This model will be needed in this part, where you will find the (only) exercise for this week. <b>We will ask you to perform and evaluate fine-tuning experiments, shifting the domain of the translation model to novels.</b>

<br/>


 <font color="blue"> <b>NOTE: </b> Before executing any code, it is a good idea to set the runtime accelerator from CPU to GPU.

You can do this via <em>Runtime > Change runtime type. </em> Select 'GPU' and then 'Save'. </font>

<br/>

If you want to run the notebook on another environment (e.g. the GPU cluster of the university) follow the instructions here to install Fairseq: https://github.com/pytorch/fairseq However note that we cannot provide extra support for this during the lab.

<br /><hr width=170% /><br />

###  <font color="blue"> <b>Please enter your name and student number below:</b></font>

Name: Marten de Jong

Student number: 4557182

<br /><hr width=170% /><br />

<h1> <font color="blue"> <b> Setup software (from PART 1)</b></h1>

We change the Python version from default 3.11 to 3.10. After executing the following command, type the selection number for Python 3.10 and press Enter.

In [ ]:
!update-alternatives --config python3

There are 2 choices for the alternative python3 (providing /usr/bin/python3).

  Selection    Path                 Priority   Status
------------------------------------------------------------
* 0            /usr/bin/python3.12   2         auto mode
  1            /usr/bin/python3.10   1         manual mode
  2            /usr/bin/python3.12   2         manual mode

Press <enter> to keep the current choice[*], or type selection number: 1
update-alternatives: using /usr/bin/python3.10 to provide /usr/bin/python3 (python3) in manual mode


In [ ]:
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py pip==24
!pip install hydra-core omegaconf bitarray sacremoses nltk sacrebleu

--2026-03-06 20:38:29--  https://bootstrap.pypa.io/get-pip.py
Resolving bootstrap.pypa.io (bootstrap.pypa.io)... 151.101.0.175, 151.101.64.175, 151.101.128.175, ...
Connecting to bootstrap.pypa.io (bootstrap.pypa.io)|151.101.0.175|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2193439 (2.1M) [text/x-python]
Saving to: ‘get-pip.py’

get-pip.py          100%[===================>]   2.09M  --.-KB/s    in 0.04s   

2026-03-06 20:38:30 (52.2 MB/s) - ‘get-pip.py’ saved [2193439/2193439]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wheel]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.7 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/pytorch/fairseq
!git clone https://github.com/rsennrich/subword-nmt
%cd fairseq
!git checkout 327cff24a57c2ae06657731bf3be86ee88fccfea
!pip install --editable ./

!pip install torch==2.5.0

%cd /content

Cloning into 'fairseq'...
remote: Enumerating objects: 35404, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 35404 (delta 17), reused 9 (delta 9), pack-reused 35357 (from 3)
Receiving objects: 100% (35404/35404), 25.50 MiB | 21.47 MiB/s, done.
Resolving deltas: 100% (25546/25546), done.
Cloning into 'subword-nmt'...
remote: Enumerating objects: 622, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 622 (delta 25), reused 31 (delta 16), pack-reused 576 (from 1)
Receiving objects: 100% (622/622), 261.27 KiB | 2.09 MiB/s, done.
Resolving deltas: 100% (374/374), done.
/content/fairseq
Note: switching to '327cff24a57c2ae06657731bf3be86ee88fccfea'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize

def my_preprocess(input_file, output_file):
  with open(input_file) as f1, open(output_file, "w") as f2:
    for line in f1:
      words = word_tokenize(line.lower())
      print(*words, file=f2)

In [ ]:
%set_env BPEROOT /content/subword-nmt/subword_nmt
import os
os.environ['PYTHONPATH'] += ":/content/fairseq/"
!echo $PYTHONPATH

env: BPEROOT=/content/subword-nmt/subword_nmt
/env/python:/content/fairseq/


In [ ]:
!nvidia-smi

Fri Mar  6 20:44:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!mkdir europarl-ted
%cd europarl-ted
!git clone https://huggingface.co/GroNLP/ik-mt-2023-europarl-ted-model
!mv ik-mt-2023-europarl-ted-model/* .
!rm -rf ik-mt-2023-europarl-ted-model/
!mv europarl-ted-transformer-model.pt europarl-ted-model.pt

/content/europarl-ted
Cloning into 'ik-mt-2023-europarl-ted-model'...
remote: Enumerating objects: 6, done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 6 (from 1)
Receiving objects: 100% (6/6), done.


In [ ]:
!git clone https://huggingface.co/datasets/GroNLP/ik-mt-2023-et-data-bin

!rm -rf et-data-bin
!mkdir -p et-data-bin
!unzip ik-mt-2023-et-data-bin/et-data-bin.zip -d et-data-bin
!rm -rf ik-mt-2023-et-data-bin/

Cloning into 'ik-mt-2023-et-data-bin'...
remote: Enumerating objects: 6, done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 6 (from 1)
Receiving objects: 100% (6/6), done.
Archive:  ik-mt-2023-et-data-bin/et-data-bin.zip
mapname:  conversion of  failed
 extracting: et-data-bin/test.nl     
 extracting: et-data-bin/test.en     
 extracting: et-data-bin/dict.nl.txt  
 extracting: et-data-bin/dict.en.txt  
 extracting: et-data-bin/preprocess.log  
 extracting: et-data-bin/test.en-nl.en.idx  
 extracting: et-data-bin/test.en-nl.nl.bin  
 extracting: et-data-bin/test.en-nl.en.bin  
 extracting: et-data-bin/test.en-nl.nl.idx  
 extracting: et-data-bin/valid.en-nl.nl.idx  
 extracting: et-data-bin/train.en-nl.nl.bin  
 extracting: et-data-bin/train.en-nl.en.idx  
 extracting: et-data-bin/valid.en-nl.en.idx  
 extracting: et-data-bin/valid.en-nl.nl.bin  
 extracting: et-data-bin/valid.en-nl.en.bin  
 extracting: et-data-bin/train.en-nl.nl.idx  
 extracting: et-data-bin/train.en-n

<h1> <font color="blue"> <b> Part III: Domain-shift exercise </b></h1>

<h2><font color=green><b>EXERCISE:</b> Domain-shift to novels</font> </h2>

In this exercise, we will explore domain-shifts in machine translation. \
More specifically, we provide you with a English-Dutch model pre-trained on a large amount of out-of-domain data (mix of parliament proceedings, Europarl, and talk transcripts, Ted) and ask you to evaluate it, and adapt it to, the translation of a novel.

While this model performs very well on a held-out set of Europarl and Ted data (28 BLEU score), you will see below that it performs very poorly when translating a novel. This is due to a very different vocabulary, but also to different writing style, different sentence length and complexity, as well as a more free (creative) translation style that is typical of literature.

Your task is to **fine-tune** the Europarl-Ted model to obtain the highest possible translation quality on a test set extracted from the book <em>Around the world in eighty days</em> by Jules Verne.\
Generally speaking, fine-tuning consists of continuing to train an already trained model on a new dataset.

You are given the following datasets:
<ul>
<li>Test set: the last 500 lines of <em>Around the world in eighty days</em> by Jules Verne.</li>
<li>Dev set: 200 non-overlapping lines of <em>Around the world in eighty days</em> by Jules Verne.</li> This is needed to monitor accuracy during fine-tuning and decide when to stop updating the model parameters.
<li>Fine-tune set 1: a portion of the same book. The first 500 lines of <em>Around the world in eighty days</em> by Jules Verne.</li>
<li>Fine-tune set 2: a different book by the same author. All 3,458 lines of <em>20,000 Leagues under the seas</em> by Jules Verne.</li>
<li>Fine-tune set 3: a book from a different author. All 10,856 lines of <em>The Three Musketeers</em> by Alexandre Dumas.</li>
</ul>


Provide an answer to each of the research questions below, by running the necessary experiments.

<font color=green><b>1A)</b></font> How does fine-tuning to a different portion of the target novel affect performance on that same target novel?

<font color=green><b>1B)</b></font> How does fine-tuning to a different novel of the same author affect performance on the target novel?

<font color=green><b>1C)</b></font> How does fine-tuning to a novel of another author affect performance on the target novel?

<font color=green><b>1D)</b></font> When controlling for fine-tuning data size (measured in number of words), which of the three data sources results in the highest test quality?

<font color=green><b>1E)</b></font> Assume you have a fixed budget of maximum 200,000 words for fine-tuning. What combination of fine-tuning data leads to the highest translation quality on the target novel?\
Tip: You can select (random) portions of the datasets, and mix them together, also randomly.

Report the scores you obtained in the table below, and then explain the answers to your research questions.

<table>
  <tr>
    <th>Model</th>
    <th>BLEU score (test)</th>
  </tr>
  <tr>
    <td>Baseline</td>
    <td>8.2866</td>
  <tr>
  <tr>
    <td>1A: same book</td>
    <td>10.6548</td>
  <tr>
    <td>1B: ...</td>
    <td>11.8907</td>
  </tr>
    <tr>
    <td>1C: ...</td>
    <td>7.4663</td>
  </tr>
    <tr>
    <td>1D: ...</td>
    <td>verne: 9.3970; dumas: 8.4650</td>


  </tr>
  </tr>
    <tr>
    <td>1E: ...</td>
    <td>13.3772</td>
  </tr>
</table>

<br />

<font color=green><b>2)</b></font> Choose one of your best fine-tuned models, and manually compare its outputs to those of the baseline (non fine-tuned) model, on a subset of the test set. \
Does fine-tuning help in any specific aspect of translation? (e.g. better vocabulary coverage, more appropriate choice of synonyms, ability to translate proper nouns, sentence length, sentence structure, punctuation... etc?)\
Provide examples to support your observations.
<br /><br /><hr width=170% /><br /><br />

The best fine-tuned model can use contextual words better than the baseline.
In the following sentence:

> Source: was there any means of detaining mr. fogg in the car , to avoid a meeting between him and the colonel ? it ought not to be a difficult task , since that gentleman was naturally sedentary and little curious . the detective , at least , seemed to have found a way ; for , after a few moments , he said to mr. fogg , `` these are long and slow hours , sir , that we are passing on the railway . ''

the baseline translates 'car' to 'auto', whilst the best model translates it to 'wagen'. I'm unsure if the best model was trending towards the use of 'wagon' but it at least did use something else than the most likely translation of 'car'.

The best model also identified the time period and translated accordingly. It used words such as 'zijne', 'eene', 'gij' 'engelsch' en 'amerikaansch'. Spelling which today is seen as old fashioned:

> 1E: `` o , maar wij kunnen gemakkelijk eenige kaarten aandoen , want zij worden op alle amerikaansche spoortrein verkocht , en als zij spelen .... ''

Certain structures such as 'neither ... nor' the best model identified with correctly translating it into old fashioned style with: 'noch ... noch'.

Certain nouns and terms the best model identified whilst the baseline lost these. The game 'whist' got lost with the baseline and the best model retained it correclty. The character 'fix' got translate to 'fix' by the best model and the baseline lost it.

There is not a significant difference in sentence length between the models. Also not a real difference in sentence structure.

In [ ]:
# Preprocess the data with Fairseq
!fairseq-preprocess --source-lang en --target-lang nl \
    --seed 12 \
    --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
    --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt  \
    --trainpref $BOOKS/Verne_ATW/train.bpe --validpref  $BOOKS/Verne_ATW/dev.bpe --testpref  $BOOKS/Verne_ATW/test.bpe \
    --destdir /content/books/finetuning/1A \
    --workers 20

# Fine-tune the model
!CUDA_VISIBLE_DEVICES=0 fairseq-train \
    /content/books/finetuning/1A \
    --source-lang en --target-lang nl \
    --finetune-from-model /content/europarl-ted/europarl-ted-model.pt \
    --seed 12 \
    --task translation \
    --arch transformer_iwslt_de_en \
    --share-decoder-input-output-embed \
    --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
    --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 \
    --dropout 0.3 --weight-decay 0.0001 \
    --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
    --batch-size 32 \
    --eval-bleu \
    --eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
    --eval-bleu-detok moses \
    --eval-bleu-remove-bpe \
    --eval-bleu-print-samples \
    --max-epoch 2 \
    --best-checkpoint-metric bleu --maximize-best-checkpoint-metric \
    --save-dir /content/finetuning-models/1A/\
    --tensorboard-logdir /content/logs/fit

# Generate translations for the test set, with our trained model
!fairseq-generate /content/books/books-data-bin/Verne_ATW --gen-subset test \
    --path /content/finetuning-models/1A/checkpoint_best.pt \
    --batch-size 64 --beam 5 \
    --remove-bpe \
    > /content/books/finetuning/1A/generate-test-verne-atw-1A.log

!grep ^H /content/books/finetuning/1A/generate-test-verne-atw-1A.log  | LC_ALL=C sort -V | cut -f3- > /content/books/finetuning/1A/generate-test-verne-atw-1A.hyp

!sacrebleu $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl -i /content/books/finetuning/1A/generate-test-verne-atw-1A.hyp -m bleu -b -w 4 --force

Traceback (most recent call last):
  File "/usr/local/bin/fairseq-preprocess", line 5, in <module>
    from fairseq_cli.preprocess import cli_main
  File "/content/fairseq/fairseq_cli/preprocess.py", line 18, in <module>
    from fairseq import options, tasks, utils
  File "/content/fairseq/fairseq/__init__.py", line 33, in <module>
    import fairseq.criterions  # noqa
  File "/content/fairseq/fairseq/criterions/__init__.py", line 36, in <module>
    importlib.import_module("fairseq.criterions." + file_name)
  File "/usr/lib/python3.10/importlib/__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "/content/fairseq/fairseq/criterions/ctc.py", line 18, in <module>
    from fairseq.data.data_utils import post_process
  File "/content/fairseq/fairseq/data/__init__.py", line 24, in <module>
    from .indexed_dataset import (
  File "/content/fairseq/fairseq/data/indexed_dataset.py", line 123, in <module>
    6: np.float,
  File "/

## First, we provide an example of fine-tuning the pre-trained model on a dummy dataset

### First, we download all preprocessed book data

This dataset is a processed subset of the Opus-Books corpus, which is itself based in  Andras Farkas' collection of copyright-free novels and their translations https://opus.nlpl.eu/Books/corpus/version/Books


In [ ]:
# Make a new directory
%cd /content
!mkdir books
%cd books

/content
/content/books


In [ ]:
# Download the book data
!git clone https://huggingface.co/datasets/GroNLP/ik-mt-2023-books
!rm -rf book_data
!mkdir -p book_data
!mv /content/books/ik-mt-2023-books/books/* book_data/
!rm -rf ik-mt-2023-books

Cloning into 'ik-mt-2023-books'...
remote: Enumerating objects: 29, done.
remote: Total 29 (delta 0), reused 0 (delta 0), pack-reused 29 (from 1)
Receiving objects: 100% (29/29), 1.73 MiB | 5.61 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
# Allow for quicker access. We see the three book folders
BOOKS="/content/books/book_data"
%set_env BOOKS /content/books/book_data

!ls $BOOKS
!head -3 $BOOKS/Verne_ATW/Verne_ATW_train*

env: BOOKS=/content/books/book_data
bpe.model-et  Dumas  Dummy  Verne_20000  Verne_ATW
==> /content/books/book_data/Verne_ATW/Verne_ATW_train.en <==
Around the World in 80 Days
Jules Verne
Chapter I IN WHICH PHILEAS FOGG AND PASSEPARTOUT ACCEPT EACH OTHER, THE ONE AS MASTER, THE OTHER AS MAN

==> /content/books/book_data/Verne_ATW/Verne_ATW_train.nl <==
De reis om de wereld in tachtig dagen
Jules Verne
EERSTE HOOFDSTUK. Waarin Phileas Fogg en Passepartout elkander wederkeerig aannemen, den een als meester, den ander als knecht.


### Tokenize and lowercase all book files

In [ ]:
# Example of naming: Verne_ATW_test.nl --> Verne_ATW_test.tok.nl
import os
dirs = [BOOKS+"/Verne_ATW", BOOKS+"/Dumas", BOOKS+"/Verne_20000", BOOKS+"/Dummy"]
for directory in dirs:
  for filename in os.listdir(directory):
      infile = os.path.join(directory, filename)
      outfile = "".join(str(infile).split(".")[0:-1]) + ".tok." + str(infile).split(".")[-1]
      my_preprocess(infile, outfile)

### Apply BPE

In [ ]:
# Apply the BPE model to all Verne Around the World files
!cat $BOOKS/Verne_ATW/Verne_ATW_test.tok.en | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/Verne_ATW/test.bpe.en
!cat $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/Verne_ATW/test.bpe.nl
!cat $BOOKS/Verne_ATW/Verne_ATW_dev.tok.en | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/Verne_ATW/dev.bpe.en
!cat $BOOKS/Verne_ATW/Verne_ATW_dev.tok.nl | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/Verne_ATW/dev.bpe.nl
!cat $BOOKS/Verne_ATW/Verne_ATW_train.tok.en | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/Verne_ATW/train.bpe.en
!cat $BOOKS/Verne_ATW/Verne_ATW_train.tok.nl | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/Verne_ATW/train.bpe.nl

In [ ]:
# again quicker access to data directory
CHANGE ="/content/fairseq/fairseq/data"
%set_env CHANGE /content/fairseq/fairseq/data

# rename to old
!mv $CHANGE/indexed_dataset.py $CHANGE/indexed_dataset_OLD.py

# paths
p1 = "/content/fairseq/fairseq/data/indexed_dataset_OLD.py"
p2 = "/content/fairseq/fairseq/data/indexed_dataset.py"

# removes np. from np.float in the file (obsolete)
with open(p1, "r") as inputfile, open(p2, "w") as outputf:
  for line in inputfile:
    if "np.float" in line:
      line = line.replace("np.", "")
    print(line, file=outputf)

env: CHANGE=/content/fairseq/fairseq/data


### We test the pre-trained model on the book data test set, without any fine-tuning




In [ ]:
!fairseq-preprocess --source-lang en --target-lang nl \
    --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
    --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt  \
    --seed 12 \
    --validpref /content/books/book_data/Verne_ATW/dev.bpe --testpref /content/books/book_data/Verne_ATW/test.bpe \
    --destdir /content/books/books-data-bin/Verne_ATW \
    --workers 20

2026-03-06 20:44:46 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:44:46 | INFO | fairseq_cli.preprocess | Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=12, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross_entropy', tokenizer=None, bpe=None, optimizer=None, lr_scheduler='f

In [ ]:
!fairseq-generate /content/books/books-data-bin/Verne_ATW --gen-subset test \
    --path /content/europarl-ted/europarl-ted-model.pt \
    --batch-size 64 --beam 5 \
    --remove-bpe \
    > $BOOKS/Verne_ATW/generate-test-verne-atw.log

2026-03-06 20:44:52 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:44:54 | INFO | fairseq_cli.generate | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 1, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': False, 'amp_batch_retries': 2, 'amp_init_scale': 128, 'amp_scale_window': None, 'user_dir': None, 'empty_cache_freq': 0, 'all_gather_list_size': 16384, 'model_parallel_size': 1, 'quantization_config_path': None, 'profile': False, 'reset_logging': False, 'suppress_crashes': False, 'use_p

In [ ]:
!head -n 20 $BOOKS/Verne_ATW/generate-test-verne-atw.log

S-33	`` on the bridge ? '' asked a passenger .
T-33	`` over de brug ? ''
H-33	-0.4525405168533325	`` aan de brug ? '' vroeg een passagiers .
D-33	-0.4525405168533325	`` aan de brug ? '' vroeg een passagiers .
P-33	-0.1358 -1.0167 -0.1387 -0.0174 -0.1538 -0.4089 -0.1425 -0.3243 -1.8696 -0.1156 -0.6546
S-270	`` i will buy it of you . ''
T-270	`` ik koop het . ''
H-270	-0.41004446148872375	`` ik zal het van jou kopen . ''
D-270	-0.41004446148872375	`` ik zal het van jou kopen . ''
P-270	-0.1218 -0.3351 -0.9434 -0.4870 -0.5405 -1.0521 -0.2252 -0.1567 -0.1066 -0.1320
S-172	`` to-morrow evening , madam . ''
T-172	`` morgen avond ; mevrouw . ''
H-172	-0.9906090497970581	`` om moraal te zijn , mevrouw . ''
D-172	-0.9906090497970581	`` om moraal te zijn , mevrouw . ''
P-172	-0.3121 -2.0351 -0.2284 -5.5038 -0.3835 -1.2775 -0.2952 -0.2807 -0.3291 -0.1151 -0.1362
S-255	`` is your vessel a swift one ? ''
T-255	`` uw schip loopt goed ? ''
H-255	-0.636360764503479	`` is je vaartuig een snelle ? ''
D-

In [ ]:
!grep ^H $BOOKS/Verne_ATW/generate-test-verne-atw.log | LC_ALL=C sort -V | cut -f3- > $BOOKS/Verne_ATW/generate-test-verne-atw.hyp
!paste -d\\n $BOOKS/Verne_ATW/Verne_ATW_test.tok.en $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl $BOOKS/Verne_ATW/generate-test-verne-atw.hyp /dev/null | head -n 40

the conversation dropped . mr. fogg had just woke up , and was looking out of the window . soon after passepartout , without being heard by his master or aouda , whispered to the detective , `` would you really fight for him ? ''
hier werd het gesprek gestaakt , daar fogg wakker werd en door het met sneeuwvlokken bedekte raampje naar het landschap staarde . eenigen tijd daarna en zonder door zijn meester of aouda gehoord te worden , zeide passepartout tot den inspecteur : `` zoudt gij waarlijk voor hem willen vechten ? ''
het gesprek daalde . meneer fogg was net wakker geworden en keek uit het raam . kort na passepartout , zonder te horen van zijn meester of aouda , die naar de detect keek , `` zou je echt voor hem vechten ? ''

`` i would do anything , '' replied fix , in a tone which betrayed determined will , `` to get him back living to europe ! ''
`` ik zal alle mogelijke moeite doen om hem levend in europa te brengen , '' antwoordde fix op een toon , die van onwrikbare kalmte get

In [ ]:
!sacrebleu $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl -i $BOOKS/Verne_ATW/generate-test-verne-atw.hyp -m bleu -b -w 4 --force

8.2919


### The score seems quite low. It is your task to try to improve this by fine-tuning the model! Below, we show an example with the dummy data. You can use this as a basis for your experiments.

In [ ]:
# Apply BPE
!cat $BOOKS/Dummy/dummy_train.tok.en | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/Dummy/train.bpe.en
!cat $BOOKS/Dummy/dummy_train.tok.nl | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/Dummy/train.bpe.nl

In [ ]:
# Preprocess the data with Fairseq
!fairseq-preprocess --source-lang en --target-lang nl \
    --seed 12 \
    --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
    --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt  \
    --trainpref $BOOKS/Dummy/train.bpe --validpref  $BOOKS/Verne_ATW/dev.bpe --testpref  $BOOKS/Verne_ATW/test.bpe \
    --destdir /content/books/finetuning/f1 \
    --workers 20

2026-03-06 20:45:18 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:45:18 | INFO | fairseq_cli.preprocess | Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=12, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross_entropy', tokenizer=None, bpe=None, optimizer=None, lr_scheduler='f

In [ ]:
# Fine-tune the model
!CUDA_VISIBLE_DEVICES=0 fairseq-train \
    /content/books/finetuning/f1 \
    --source-lang en --target-lang nl \
    --finetune-from-model /content/europarl-ted/europarl-ted-model.pt \
    --seed 12 \
    --task translation \
    --arch transformer_iwslt_de_en \
    --share-decoder-input-output-embed \
    --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
    --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 \
    --dropout 0.3 --weight-decay 0.0001 \
    --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
    --batch-size 32 \
    --eval-bleu \
    --eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
    --eval-bleu-detok moses \
    --eval-bleu-remove-bpe \
    --eval-bleu-print-samples \
    --max-epoch 2 \
    --best-checkpoint-metric bleu --maximize-best-checkpoint-metric \
    --save-dir /content/finetuning-models/f1/\
    --tensorboard-logdir /content/logs/fit

2026-03-06 20:45:24 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:45:26 | INFO | fairseq_cli.train | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'tensorboard_logdir': '/content/logs/fit', 'wandb_project': None, 'azureml_logging': False, 'seed': 12, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': False, 'amp_batch_retries': 2, 'amp_init_scale': 128, 'amp_scale_window': None, 'user_dir': None, 'empty_cache_freq': 0, 'all_gather_list_size': 16384, 'model_parallel_size': 1, 'quantization_config_path': None, 'profile': False, 'reset_logging': False, 'suppress_crashes': 

In [ ]:
# Generate translations for the test set, with our trained model
!fairseq-generate /content/books/books-data-bin/Verne_ATW --gen-subset test \
    --path /content/finetuning-models/f1/checkpoint_best.pt \
    --batch-size 64 --beam 5 \
    --remove-bpe \
    > /content/books/finetuning/f1/generate-test-verne-atw-f1.log

2026-03-06 20:46:14 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:46:15 | INFO | fairseq_cli.generate | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 1, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': False, 'amp_batch_retries': 2, 'amp_init_scale': 128, 'amp_scale_window': None, 'user_dir': None, 'empty_cache_freq': 0, 'all_gather_list_size': 16384, 'model_parallel_size': 1, 'quantization_config_path': None, 'profile': False, 'reset_logging': False, 'suppress_crashes': False, 'use_p

In [ ]:
# Prepare file for evaluation
!grep ^H /content/books/finetuning/f1/generate-test-verne-atw-f1.log  | LC_ALL=C sort -V | cut -f3- > /content/books/finetuning/f1/generate-test-verne-atw-f1.hyp

In [ ]:
# Get the BLEU score
!sacrebleu $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl -i /content/books/finetuning/f1/generate-test-verne-atw-f1.hyp -m bleu -b -w 4 --force

8.2901


### Now it's your turn!

# 1A

In [ ]:
# Preprocess the data with Fairseq
!fairseq-preprocess --source-lang en --target-lang nl \
    --seed 12 \
    --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
    --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt  \
    --trainpref $BOOKS/Verne_ATW/train.bpe --validpref  $BOOKS/Verne_ATW/dev.bpe --testpref  $BOOKS/Verne_ATW/test.bpe \
    --destdir /content/books/finetuning/1A \
    --workers 20

# Fine-tune the model
!CUDA_VISIBLE_DEVICES=0 fairseq-train \
    /content/books/finetuning/1A \
    --source-lang en --target-lang nl \
    --finetune-from-model /content/europarl-ted/europarl-ted-model.pt \
    --seed 12 \
    --task translation \
    --arch transformer_iwslt_de_en \
    --share-decoder-input-output-embed \
    --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
    --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 \
    --dropout 0.3 --weight-decay 0.0001 \
    --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
    --batch-size 32 \
    --eval-bleu \
    --eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
    --eval-bleu-detok moses \
    --eval-bleu-remove-bpe \
    --eval-bleu-print-samples \
    --max-epoch 2 \
    --best-checkpoint-metric bleu --maximize-best-checkpoint-metric \
    --save-dir /content/finetuning-models/1A/\
    --tensorboard-logdir /content/logs/fit

# Generate translations for the test set, with our trained model
!fairseq-generate /content/books/books-data-bin/Verne_ATW --gen-subset test \
    --path /content/finetuning-models/1A/checkpoint_best.pt \
    --batch-size 64 --beam 5 \
    --remove-bpe \
    > /content/books/finetuning/1A/generate-test-verne-atw-1A.log

!grep ^H /content/books/finetuning/1A/generate-test-verne-atw-1A.log  | LC_ALL=C sort -V | cut -f3- > /content/books/finetuning/1A/generate-test-verne-atw-1A.hyp

!sacrebleu $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl -i /content/books/finetuning/1A/generate-test-verne-atw-1A.hyp -m bleu -b -w 4 --force

2026-03-06 20:46:39 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:46:39 | INFO | fairseq_cli.preprocess | Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=12, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross_entropy', tokenizer=None, bpe=None, optimizer=None, lr_scheduler='f

In [ ]:
!rm -rf /content/finetuning-models/1A

Fine-tuning the model on a different portion of the same novel increased the score from 8.3 to 10.7. The model adapted the specific vocabulary and writing style of the target text.

# 1B

In [ ]:
%env ARGUMENT=1B
%env BOOK=Verne_20000
%env TRAIN=Verne_20000_train

# Apply BPE
!cat $BOOKS/$BOOK/$TRAIN.tok.en | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/$BOOK/$TRAIN.bpe.en
!cat $BOOKS/$BOOK/$TRAIN.tok.nl | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/$BOOK/$TRAIN.bpe.nl

# Preprocess the data with Fairseq
!fairseq-preprocess --source-lang en --target-lang nl \
    --seed 12 \
    --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
    --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt  \
    --trainpref $BOOKS/$BOOK/$TRAIN.bpe --validpref  $BOOKS/Verne_ATW/dev.bpe --testpref  $BOOKS/Verne_ATW/test.bpe \
    --destdir /content/books/finetuning/$ARGUMENT \
    --workers 20

# Fine-tune the model
!CUDA_VISIBLE_DEVICES=0 fairseq-train \
    /content/books/finetuning/$ARGUMENT \
    --source-lang en --target-lang nl \
    --finetune-from-model /content/europarl-ted/europarl-ted-model.pt \
    --seed 12 \
    --task translation \
    --arch transformer_iwslt_de_en \
    --share-decoder-input-output-embed \
    --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
    --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 \
    --dropout 0.3 --weight-decay 0.0001 \
    --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
    --batch-size 32 \
    --eval-bleu \
    --eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
    --eval-bleu-detok moses \
    --eval-bleu-remove-bpe \
    --eval-bleu-print-samples \
    --max-epoch 2 \
    --best-checkpoint-metric bleu --maximize-best-checkpoint-metric \
    --save-dir /content/finetuning-models/$ARGUMENT/ \
    --tensorboard-logdir /content/logs/fit

# Generate translations for the test set, with our trained model
!fairseq-generate /content/books/books-data-bin/Verne_ATW --gen-subset test \
    --path /content/finetuning-models/$ARGUMENT/checkpoint_best.pt \
    --batch-size 64 --beam 5 \
    --remove-bpe \
    > /content/books/finetuning/$ARGUMENT/generate-test-verne-atw-$ARGUMENT.log

!grep ^H /content/books/finetuning/$ARGUMENT/generate-test-verne-atw-$ARGUMENT.log  | LC_ALL=C sort -V | cut -f3- > /content/books/finetuning/$ARGUMENT/generate-test-verne-atw-$ARGUMENT.hyp

!sacrebleu $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl -i /content/books/finetuning/$ARGUMENT/generate-test-verne-atw-$ARGUMENT.hyp -m bleu -b -w 4 --force

env: ARGUMENT=1B
env: BOOK=Verne_20000
env: TRAIN=Verne_20000_train
2026-03-06 20:48:09 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:48:10 | INFO | fairseq_cli.preprocess | Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=12, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross

In [ ]:
!rm -rf /content/finetuning-models/1B

Including different data but still from the same author increased the score again to 11.9. The writing style is very similar therefore contributing positively to the ability to translate.

# 1C

In [ ]:
%env ARGUMENT=1C
%env BOOK=Dumas
%env TRAIN=Dumas_TTM_train

# Apply BPE
!cat $BOOKS/$BOOK/$TRAIN.tok.en | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/$BOOK/$TRAIN.bpe.en
!cat $BOOKS/$BOOK/$TRAIN.tok.nl | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/$BOOK/$TRAIN.bpe.nl

# Preprocess the data with Fairseq
!fairseq-preprocess --source-lang en --target-lang nl \
    --seed 12 \
    --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
    --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt  \
    --trainpref $BOOKS/$BOOK/$TRAIN.bpe --validpref  $BOOKS/Verne_ATW/dev.bpe --testpref  $BOOKS/Verne_ATW/test.bpe \
    --destdir /content/books/finetuning/$ARGUMENT \
    --workers 20

# Fine-tune the model
!CUDA_VISIBLE_DEVICES=0 fairseq-train \
    /content/books/finetuning/$ARGUMENT \
    --source-lang en --target-lang nl \
    --finetune-from-model /content/europarl-ted/europarl-ted-model.pt \
    --seed 12 \
    --task translation \
    --arch transformer_iwslt_de_en \
    --share-decoder-input-output-embed \
    --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
    --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 \
    --dropout 0.3 --weight-decay 0.0001 \
    --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
    --batch-size 32 \
    --eval-bleu \
    --eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
    --eval-bleu-detok moses \
    --eval-bleu-remove-bpe \
    --eval-bleu-print-samples \
    --max-epoch 2 \
    --best-checkpoint-metric bleu --maximize-best-checkpoint-metric \
    --save-dir /content/finetuning-models/$ARGUMENT/ \
    --tensorboard-logdir /content/logs/fit

# Generate translations for the test set, with our trained model
!fairseq-generate /content/books/books-data-bin/Verne_ATW --gen-subset test \
    --path /content/finetuning-models/$ARGUMENT/checkpoint_best.pt \
    --batch-size 64 --beam 5 \
    --remove-bpe \
    > /content/books/finetuning/$ARGUMENT/generate-test-verne-atw-$ARGUMENT.log

!grep ^H /content/books/finetuning/$ARGUMENT/generate-test-verne-atw-$ARGUMENT.log  | LC_ALL=C sort -V | cut -f3- > /content/books/finetuning/$ARGUMENT/generate-test-verne-atw-$ARGUMENT.hyp

!sacrebleu $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl -i /content/books/finetuning/$ARGUMENT/generate-test-verne-atw-$ARGUMENT.hyp -m bleu -b -w 4 --force

env: ARGUMENT=1C
env: BOOK=Dumas
env: TRAIN=Dumas_TTM_train
2026-03-06 20:50:29 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:50:29 | INFO | fairseq_cli.preprocess | Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=12, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross_entropy

In [ ]:
!rm -rf /content/finetuning-models/1C

Using a different book from another author, clearly with a different style and vocabulary because the score went down significantly, shows that the model loses translation capabilities without gathering any relevant knowledge present in the domain.

# 1D

In [ ]:
!mkdir -p $BOOKS/Verne_20000_500
!mkdir -p $BOOKS/Dumas_500

!head -n 500 $BOOKS/Verne_20000/Verne_20000_train.tok.en > $BOOKS/Verne_20000_500/train.tok.en
!head -n 500 $BOOKS/Verne_20000/Verne_20000_train.tok.nl > $BOOKS/Verne_20000_500/train.tok.nl

!head -n 500 $BOOKS/Dumas/Dumas_TTM_train.tok.en > $BOOKS/Dumas_500/train.tok.en
!head -n 500 $BOOKS/Dumas/Dumas_TTM_train.tok.nl > $BOOKS/Dumas_500/train.tok.nl

In [ ]:
%env ARGUMENT=1D_Verne
%env BOOK=Verne_20000_500
%env TRAIN=train

# Apply BPE
!cat $BOOKS/$BOOK/$TRAIN.tok.en | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/$BOOK/$TRAIN.bpe.en
!cat $BOOKS/$BOOK/$TRAIN.tok.nl | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/$BOOK/$TRAIN.bpe.nl

# Preprocess the data with Fairseq
!fairseq-preprocess --source-lang en --target-lang nl \
    --seed 12 \
    --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
    --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt  \
    --trainpref $BOOKS/$BOOK/$TRAIN.bpe --validpref  $BOOKS/Verne_ATW/dev.bpe --testpref  $BOOKS/Verne_ATW/test.bpe \
    --destdir /content/books/finetuning/$ARGUMENT \
    --workers 20

# Fine-tune the model
!CUDA_VISIBLE_DEVICES=0 fairseq-train \
    /content/books/finetuning/$ARGUMENT \
    --source-lang en --target-lang nl \
    --finetune-from-model /content/europarl-ted/europarl-ted-model.pt \
    --seed 12 \
    --task translation \
    --arch transformer_iwslt_de_en \
    --share-decoder-input-output-embed \
    --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
    --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 \
    --dropout 0.3 --weight-decay 0.0001 \
    --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
    --batch-size 32 \
    --eval-bleu \
    --eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
    --eval-bleu-detok moses \
    --eval-bleu-remove-bpe \
    --eval-bleu-print-samples \
    --max-epoch 2 \
    --best-checkpoint-metric bleu --maximize-best-checkpoint-metric \
    --save-dir /content/finetuning-models/$ARGUMENT/ \
    --tensorboard-logdir /content/logs/fit

# Generate translations for the test set, with our trained model
!fairseq-generate /content/books/books-data-bin/Verne_ATW --gen-subset test \
    --path /content/finetuning-models/$ARGUMENT/checkpoint_best.pt \
    --batch-size 64 --beam 5 \
    --remove-bpe \
    > /content/books/finetuning/$ARGUMENT/generate-test-$ARGUMENT.log

!grep ^H /content/books/finetuning/$ARGUMENT/generate-test-$ARGUMENT.log  | LC_ALL=C sort -V | cut -f3- > /content/books/finetuning/$ARGUMENT/generate-test-$ARGUMENT.hyp

!sacrebleu $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl -i /content/books/finetuning/$ARGUMENT/generate-test-$ARGUMENT.hyp -m bleu -b -w 4 --force

env: ARGUMENT=1D_Verne
env: BOOK=Verne_20000_500
env: TRAIN=train
2026-03-06 20:53:50 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:53:51 | INFO | fairseq_cli.preprocess | Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=12, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross_e

In [ ]:
%env ARGUMENT=1D_Dumas
%env BOOK=Dumas_500
%env TRAIN=train

# Apply BPE
!cat $BOOKS/$BOOK/$TRAIN.tok.en | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/$BOOK/$TRAIN.bpe.en
!cat $BOOKS/$BOOK/$TRAIN.tok.nl | python $BPEROOT/apply_bpe.py --codes $BOOKS/bpe.model-et > $BOOKS/$BOOK/$TRAIN.bpe.nl

# Preprocess the data with Fairseq
!fairseq-preprocess --source-lang en --target-lang nl \
    --seed 12 \
    --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
    --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt  \
    --trainpref $BOOKS/$BOOK/$TRAIN.bpe --validpref  $BOOKS/Verne_ATW/dev.bpe --testpref  $BOOKS/Verne_ATW/test.bpe \
    --destdir /content/books/finetuning/$ARGUMENT \
    --workers 20

# Fine-tune the model
!CUDA_VISIBLE_DEVICES=0 fairseq-train \
    /content/books/finetuning/$ARGUMENT \
    --source-lang en --target-lang nl \
    --finetune-from-model /content/europarl-ted/europarl-ted-model.pt \
    --seed 12 \
    --task translation \
    --arch transformer_iwslt_de_en \
    --share-decoder-input-output-embed \
    --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
    --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 \
    --dropout 0.3 --weight-decay 0.0001 \
    --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
    --batch-size 32 \
    --eval-bleu \
    --eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
    --eval-bleu-detok moses \
    --eval-bleu-remove-bpe \
    --eval-bleu-print-samples \
    --max-epoch 2 \
    --best-checkpoint-metric bleu --maximize-best-checkpoint-metric \
    --save-dir /content/finetuning-models/$ARGUMENT/ \
    --tensorboard-logdir /content/logs/fit

# Generate translations for the test set, with our trained model
!fairseq-generate /content/books/books-data-bin/Verne_ATW --gen-subset test \
    --path /content/finetuning-models/$ARGUMENT/checkpoint_best.pt \
    --batch-size 64 --beam 5 \
    --remove-bpe \
    > /content/books/finetuning/$ARGUMENT/generate-test-$ARGUMENT.log

!grep ^H /content/books/finetuning/$ARGUMENT/generate-test-$ARGUMENT.log  | LC_ALL=C sort -V | cut -f3- > /content/books/finetuning/$ARGUMENT/generate-test-$ARGUMENT.hyp

!sacrebleu $BOOKS/Verne_ATW/Verne_ATW_test.tok.nl -i /content/books/finetuning/$ARGUMENT/generate-test-$ARGUMENT.hyp -m bleu -b -w 4 --force

env: ARGUMENT=1D_Dumas
env: BOOK=Dumas_500
env: TRAIN=train
2026-03-06 20:55:18 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 20:55:18 | INFO | fairseq_cli.preprocess | Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=12, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross_entropy

In [ ]:
!rm -rf /content/finetuning-models/1D_Dumas/
!rm -rf /content/finetuning-models/1D_Verne/

Author style is more important than just using any 'novel'. The verne model perfomred better than the dumas model.

# 1E

In [ ]:
!wc -w $BOOKS/Verne_ATW/*.tok.*
!wc -w $BOOKS/Verne_20000/Verne_20000_train.tok.*
!wc -w $BOOKS/Dumas/Dumas_TTM_train.tok.*

 10294 /content/books/book_data/Verne_ATW/Verne_ATW_dev.tok.en
 11358 /content/books/book_data/Verne_ATW/Verne_ATW_dev.tok.nl
 19532 /content/books/book_data/Verne_ATW/Verne_ATW_test.tok.en
 20372 /content/books/book_data/Verne_ATW/Verne_ATW_test.tok.nl
 22946 /content/books/book_data/Verne_ATW/Verne_ATW_train.tok.en
 25515 /content/books/book_data/Verne_ATW/Verne_ATW_train.tok.nl
110017 total
 164162 /content/books/book_data/Verne_20000/Verne_20000_train.tok.en
 154890 /content/books/book_data/Verne_20000/Verne_20000_train.tok.nl
 319052 total
 277509 /content/books/book_data/Dumas/Dumas_TTM_train.tok.en
 293590 /content/books/book_data/Dumas/Dumas_TTM_train.tok.nl
 571099 total


In [ ]:
!rm -rf /content/books/finetuning/1E
!rm -rf /content/finetuning-models/1E/

In [ ]:
%env ARGUMENT=1E
%env BOOK=Verne_1E
%env TRAIN=train

!mkdir -p $BOOKS/$BOOK

!head -n 1600 $BOOKS/Verne_20000/Verne_20000_train.tok.en > $BOOKS/$BOOK/Verne_20000_1600.tok.en
!head -n 1600 $BOOKS/Verne_20000/Verne_20000_train.tok.nl > $BOOKS/$BOOK/Verne_20000_1600.tok.nl

!cat $BOOKS/Verne_ATW/Verne_ATW_train.tok.en $BOOKS/$BOOK/Verne_20000_1600.tok.en > $BOOKS/$BOOK/$TRAIN.tok.en
!cat $BOOKS/Verne_ATW/Verne_ATW_train.tok.nl $BOOKS/$BOOK/Verne_20000_1600.tok.nl > $BOOKS/$BOOK/$TRAIN.tok.nl

!wc -w $BOOKS/$BOOK/$TRAIN.tok.*

env: ARGUMENT=1E
env: BOOK=Verne_1E
env: TRAIN=train
  97991 /content/books/book_data/Verne_1E/train.tok.en
  96677 /content/books/book_data/Verne_1E/train.tok.nl
 194668 total


In [ ]:
!fairseq-preprocess --source-lang en --target-lang nl \
  --seed 12 \
  --srcdict /content/europarl-ted/et-data-bin/dict.en.txt \
  --tgtdict /content/europarl-ted/et-data-bin/dict.nl.txt \
  --trainpref $BOOKS/1E_mixed/train.bpe \
  --validpref $BOOKS/Verne_ATW/dev.bpe \
  --testpref $BOOKS/Verne_ATW/test.bpe \
  --destdir /content/books/finetuning/1E \
  --workers 20

2026-03-06 21:07:47 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 21:07:47 | INFO | fairseq_cli.preprocess | Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=12, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross_entropy', tokenizer=None, bpe=None, optimizer=None, lr_scheduler='f

In [ ]:
!CUDA_VISIBLE_DEVICES=0 fairseq-train \
  /content/books/finetuning/1E \
  --source-lang en --target-lang nl \
  --finetune-from-model /content/europarl-ted/europarl-ted-model.pt \
  --seed 12 \
  --task translation \
  --arch transformer_iwslt_de_en \
  --share-decoder-input-output-embed \
  --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
  --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 \
  --dropout 0.3 --weight-decay 0.0001 \
  --criterion label_smoothed_cross_entropy --label-smoothing 0.1 \
  --batch-size 32 \
  --eval-bleu \
  --eval-bleu-args '{"beam": 5, "max_len_a": 1.2, "max_len_b": 10}' \
  --eval-bleu-detok moses \
  --eval-bleu-remove-bpe '@@ ' \
  --eval-bleu-print-samples \
  --best-checkpoint-metric bleu --maximize-best-checkpoint-metric \
  --max-epoch 10 \
  --save-dir /content/finetuning-models/1E/ \
  --no-epoch-checkpoints

2026-03-06 21:08:02 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 21:08:04 | INFO | fairseq_cli.train | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 12, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': False, 'amp_batch_retries': 2, 'amp_init_scale': 128, 'amp_scale_window': None, 'user_dir': None, 'empty_cache_freq': 0, 'all_gather_list_size': 16384, 'model_parallel_size': 1, 'quantization_config_path': None, 'profile': False, 'reset_logging': False, 'suppress_crashes': False, 'use_pla

In [ ]:
!fairseq-generate /content/books/finetuning/1E --gen-subset test \
  --path /content/finetuning-models/1E/checkpoint_best.pt \
  --batch-size 64 --beam 5 \
  --remove-bpe \
  > /content/books/finetuning/1E/generate-test-1E.log

!grep ^H /content/books/finetuning/1E/generate-test-1E.log | cut -f3- > /content/books/finetuning/1E/hyp.nl
!grep ^T /content/books/finetuning/1E/generate-test-1E.log | cut -f2- > /content/books/finetuning/1E/ref.nl

!sacrebleu /content/books/finetuning/1E/ref.nl -i /content/books/finetuning/1E/hyp.nl -m bleu -b -w 4

2026-03-06 21:13:54 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-03-06 21:13:55 | INFO | fairseq_cli.generate | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 1, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': False, 'amp_batch_retries': 2, 'amp_init_scale': 128, 'amp_scale_window': None, 'user_dir': None, 'empty_cache_freq': 0, 'all_gather_list_size': 16384, 'model_parallel_size': 1, 'quantization_config_path': None, 'profile': False, 'reset_logging': False, 'suppress_crashes': False, 'use_p

# 2

In [ ]:
!grep ^H /content/books/finetuning/1E/generate-test-1E.log | LC_ALL=C sort -V | cut -f3- > /content/books/finetuning/1E/generate-test-1E.hyp

!paste -d '\n' \
    $BOOKS/Verne_ATW/Verne_ATW_test.tok.en \
    $BOOKS/Verne_ATW/generate-test-verne-atw.hyp \
    /content/books/finetuning/1E/generate-test-1E.hyp \
    | head -n 30 | awk 'NR%3==1 {print "Source: " $0} NR%3==2 {print "Baseline: " $0} NR%3==0 {print "1E: " $0 "\n"}'

Source: the conversation dropped . mr. fogg had just woke up , and was looking out of the window . soon after passepartout , without being heard by his master or aouda , whispered to the detective , `` would you really fight for him ? ''
Baseline: het gesprek daalde . meneer fogg was net wakker geworden en keek uit het raam . kort na passepartout , zonder te horen van zijn meester of aouda , die naar de detect keek , `` zou je echt voor hem vechten ? ''
1E: het gesprek viel af . de heer fogg was zoo wakker geworden , en keek uit het raam . weldra na passepartout , zonder van zijn meester of aouda gehoord te worden , vroeg hij naar den detective , `` zou gij voor hem vechten ? ''

Source: `` i would do anything , '' replied fix , in a tone which betrayed determined will , `` to get him back living to europe ! ''
Baseline: `` ik zou iets doen , '' antwoordde , in een toon die vastberaden wil , `` om hem terug te brengen in europa ! ''
1E: `` ik doe wat dan ook , '' antwoordde fix , op ee

2) Choose one of your best fine-tuned models, and manually compare its outputs to those of the baseline (non fine-tuned) model, on a subset of the test set.
Does fine-tuning help in any specific aspect of translation? (e.g. better vocabulary coverage, more appropriate choice of synonyms, ability to translate proper nouns, sentence length, sentence structure, punctuation... etc?)
Provide examples to support your observations.


The best fine-tuned model can use contextual words better than the baseline.
In the following sentence:

> Source: was there any means of detaining mr. fogg in the car , to avoid a meeting between him and the colonel ? it ought not to be a difficult task , since that gentleman was naturally sedentary and little curious . the detective , at least , seemed to have found a way ; for , after a few moments , he said to mr. fogg , `` these are long and slow hours , sir , that we are passing on the railway . ''

the baseline translates 'car' to 'auto', whilst the best model translates it to 'wagen'. I'm unsure if the best model was trending towards the use of 'wagon' but it at least did use something else than the most likely translation of 'car'.

The best model also identified the time period and translated accordingly. It used words such as 'zijne', 'eene', 'gij' 'engelsch' en 'amerikaansch'. Spelling which today is seen as old fashioned:

> 1E: `` o , maar wij kunnen gemakkelijk eenige kaarten aandoen , want zij worden op alle amerikaansche spoortrein verkocht , en als zij spelen .... ''

Certain structures such as 'neither ... nor' the best model identified with correctly translating it into old fashioned style with: 'noch ... noch'.

Certain nouns and terms the best model identified whilst the baseline lost these. The game 'whist' got lost with the baseline and the best model retained it correclty. The character 'fix' got translate to 'fix' by the best model and the baseline lost it.

There is not a significant difference in sentence length between the models. Also not a real difference in sentence structure.

In [ ]:
# !rm -rf /content/books/finetuning/*
# !rm -rf /finetuning-models/*